# Homework

## Dataset

```bash
wget https://github.com/SVizor42/ML_Zoomcamp/releases/download/straight-curly-data/data.zip
unzip data.zip
```

## Reproducibility

In [ ]:
import numpy as np
import torch

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## Model

In [3]:
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
import os
from torch.utils.data import Dataset
from PIL import Image

class HairDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.classes = sorted(os.listdir(data_dir))
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}

        for label_name in self.classes:
            label_dir = os.path.join(data_dir, label_name)
            for img_name in os.listdir(label_dir):
                self.image_paths.append(os.path.join(label_dir, img_name))
                self.labels.append(self.class_to_idx[label_name])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

In [4]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        
        # Conv2D: 32 filters, kernel 3x3, activation relu
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3)
        
        # MaxPooling2D: 2x2
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Flatten happens in forward()
        
        # After Conv + Pool, calculate output size:
        # Input: (3, 200, 200)
        # Conv3x3 ⇒ (32, 198, 198)
        # MaxPool2x2 ⇒ (32, 99, 99)
        self.flatten_size = 32 * 99 * 99

        # Dense layer: 64 neurons + relu
        self.fc1 = nn.Linear(self.flatten_size, 64)
        
        # Output layer: 1 neuron
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        
        x = x.view(x.size(0), -1)  # Flatten
        
        x = F.relu(self.fc1(x))
        
        # No sigmoid here! BCEWithLogitsLoss expects raw logits
        x = self.fc2(x)
        return x

In [5]:
criterion = nn.BCEWithLogitsLoss()

In [6]:
model = SimpleCNN()
optimizer = torch.optim.SGD(model.parameters(), lr=0.002, momentum=0.8)

## Question 1

Which loss function you will use?

- nn.MSELoss()
- nn.BCEWithLogitsLoss()
- nn.CrossEntropyLoss()
- nn.CosineEmbeddingLoss()

A/ `nn.BCEWithLogitsLoss()`


## Question 2

What's the total number of parameters of the model? You can use torchsummary or count manually.

- 896
- 11214912
- 15896912
- 20073473


In [7]:
# Option 1: Using torchsummary (install with: pip install torchsummary)
from torchsummary import summary
summary(model, input_size=(3, 200, 200))

# Option 2: Manual counting
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params}")

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 198, 198]             896
         MaxPool2d-2           [-1, 32, 99, 99]               0
            Linear-3                   [-1, 64]      20,072,512
            Linear-4                    [-1, 1]              65
Total params: 20,073,473
Trainable params: 20,073,473
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.46
Forward/backward pass size (MB): 11.96
Params size (MB): 76.57
Estimated Total Size (MB): 89.00
----------------------------------------------------------------
Total parameters: 20073473


A/ `20073473`

## Generators and Training

In [8]:
from torch.utils.data import DataLoader
from torchvision import transforms

In [9]:
train_transforms = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ) # ImageNet normalization
])

val_transforms = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ) # ImageNet normalization
])

In [10]:
train_dataset = HairDataset(
    data_dir='./data/train',
    transform=train_transforms
)

validation_dataset = HairDataset(
    data_dir='./data/test',
    transform=val_transforms
)

train_loader = DataLoader(train_dataset, batch_size=20, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=20, shuffle=False)

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [12]:
num_epochs = 10
history = {'acc': [], 'loss': [], 'val_acc': [], 'val_loss': []}

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        labels = labels.float().unsqueeze(1) # Ensure labels are float and have shape (batch_size, 1)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        # For binary classification with BCEWithLogitsLoss, apply sigmoid to outputs before thresholding for accuracy
        predicted = (torch.sigmoid(outputs) > 0.5).float()
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = correct_train / total_train
    history['loss'].append(epoch_loss)
    history['acc'].append(epoch_acc)

    model.eval()
    val_running_loss = 0.0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for images, labels in validation_loader:
            images, labels = images.to(device), labels.to(device)
            labels = labels.float().unsqueeze(1)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_epoch_loss = val_running_loss / len(validation_dataset)
    val_epoch_acc = correct_val / total_val
    history['val_loss'].append(val_epoch_loss)
    history['val_acc'].append(val_epoch_acc)

    print(f"Epoch {epoch+1}/{num_epochs}, "
          f"Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}, "
          f"Val Loss: {val_epoch_loss:.4f}, Val Acc: {val_epoch_acc:.4f}")

Epoch 1/10, Loss: 0.6477, Acc: 0.6255, Val Loss: 0.9352, Val Acc: 0.5323
Epoch 2/10, Loss: 0.6055, Acc: 0.6679, Val Loss: 0.6306, Val Acc: 0.6169
Epoch 3/10, Loss: 0.5788, Acc: 0.6792, Val Loss: 0.6571, Val Acc: 0.6517
Epoch 4/10, Loss: 0.5082, Acc: 0.7441, Val Loss: 0.7942, Val Acc: 0.6070
Epoch 5/10, Loss: 0.4876, Acc: 0.7591, Val Loss: 0.6348, Val Acc: 0.6517
Epoch 6/10, Loss: 0.4183, Acc: 0.7978, Val Loss: 1.0846, Val Acc: 0.5672
Epoch 7/10, Loss: 0.4799, Acc: 0.7541, Val Loss: 0.6462, Val Acc: 0.6269
Epoch 8/10, Loss: 0.3971, Acc: 0.8177, Val Loss: 0.6494, Val Acc: 0.6965
Epoch 9/10, Loss: 0.3322, Acc: 0.8527, Val Loss: 0.7050, Val Acc: 0.6716
Epoch 10/10, Loss: 0.3198, Acc: 0.8589, Val Loss: 0.8156, Val Acc: 0.6667


## Question 3
What is the median of training accuracy for all the epochs for this model?

- 0.05
- 0.12
- 0.40
- 0.84


In [13]:
round(np.median(history['acc']), 2)

np.float64(0.76)

A/ `0.76`

## Question 4
What is the standard deviation of training loss for all the epochs for this model?

- 0.007
- 0.078
- 0.171
- 1.710


In [14]:
round(np.std(history['loss']), 3)

np.float64(0.106)

A/ `0.106`

## Data Augmentation

In [15]:
augmented_train_transforms = transforms.Compose([
    transforms.RandomRotation(50),
    transforms.RandomResizedCrop(200, scale=(0.9, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(),
    transforms.Resize((200, 200)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ), # ImageNet normalization
])

In [16]:
augmented_train_dataset = HairDataset(
    data_dir='./data/train',
    transform=train_transforms
)

augmented_train_loader = DataLoader(augmented_train_dataset, batch_size=20, shuffle=True)

## Question 5
Let's train our model for 10 more epochs using the same code as previously.

> Note: make sure you don't re-create the model. we want to continue training the model we already started training.

What is the mean of test loss for all the epochs for the model trained with augmentations?

- 0.008
- 0.08
- 0.88
- 8.88


In [17]:
num_epochs = 10
# history = {'acc': [], 'loss': [], 'val_acc': [], 'val_loss': []}

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    for images, labels in augmented_train_loader:
        images, labels = images.to(device), labels.to(device)
        labels = labels.float().unsqueeze(1) # Ensure labels are float and have shape (batch_size, 1)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        # For binary classification with BCEWithLogitsLoss, apply sigmoid to outputs before thresholding for accuracy
        predicted = (torch.sigmoid(outputs) > 0.5).float()
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = correct_train / total_train
    history['loss'].append(epoch_loss)
    history['acc'].append(epoch_acc)

    model.eval()
    val_running_loss = 0.0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for images, labels in validation_loader:
            images, labels = images.to(device), labels.to(device)
            labels = labels.float().unsqueeze(1)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_epoch_loss = val_running_loss / len(validation_dataset)
    val_epoch_acc = correct_val / total_val
    history['val_loss'].append(val_epoch_loss)
    history['val_acc'].append(val_epoch_acc)

    print(f"Epoch {epoch+1}/{num_epochs}, "
          f"Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}, "
          f"Val Loss: {val_epoch_loss:.4f}, Val Acc: {val_epoch_acc:.4f}")

Epoch 1/10, Loss: 0.3473, Acc: 0.8539, Val Loss: 0.7637, Val Acc: 0.6716
Epoch 2/10, Loss: 0.2017, Acc: 0.9288, Val Loss: 0.7968, Val Acc: 0.6617
Epoch 3/10, Loss: 0.1595, Acc: 0.9488, Val Loss: 0.8644, Val Acc: 0.6766
Epoch 4/10, Loss: 0.0947, Acc: 0.9750, Val Loss: 1.8234, Val Acc: 0.5871
Epoch 5/10, Loss: 0.1681, Acc: 0.9363, Val Loss: 0.8652, Val Acc: 0.7662
Epoch 6/10, Loss: 0.0944, Acc: 0.9763, Val Loss: 0.8108, Val Acc: 0.7662
Epoch 7/10, Loss: 0.0471, Acc: 0.9938, Val Loss: 1.1285, Val Acc: 0.7015
Epoch 8/10, Loss: 0.1983, Acc: 0.9326, Val Loss: 0.8926, Val Acc: 0.7214
Epoch 9/10, Loss: 0.0422, Acc: 0.9938, Val Loss: 1.0867, Val Acc: 0.7114
Epoch 10/10, Loss: 0.0268, Acc: 1.0000, Val Loss: 1.0445, Val Acc: 0.7463


In [18]:
round(np.mean(history['val_loss']), 2)

np.float64(0.88)

A/ `0.88`

## Question 6
What's the average of test accuracy for the last 5 epochs (from 6 to 10) for the model trained with augmentations?

- 0.08
- 0.28
- 0.68
- 0.98

In [19]:
round(np.average(history['val_acc']), 2)

np.float64(0.66)

A/ `0.66`